In [ ]:
# Uninstall the PEFT library if it causes compatibility issues with transformers
# (e.g., if you're not using LoRA or adapter-based fine-tuning)
!pip uninstall -y peft

In [ ]:
# Install specific PyTorch + CUDA 12.1 compatible versions
# (This matches with Colab's A100 GPUs and avoids binary mismatches)
!pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121 -q

# Uninstall any potentially incompatible versions of transformers and accelerate
!pip uninstall -y transformers accelerate

# Install stable, compatible versions of Hugging Face libraries
!pip install transformers==4.41.2 accelerate==0.30.1 datasets evaluate rouge_score -q

# Install utility for handling JSON Lines (used in your datasets)
!pip install jsonlines


In [ ]:
# Install sacremoses, a dependency for some tokenizers
!pip install sacremoses

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Import core libraries
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    EarlyStoppingCallback
)
from datasets import load_dataset, load_from_disk, Dataset
import evaluate

# Utilities
import pandas as pd
import matplotlib.pyplot as plt
import jsonlines
import numpy as np
from tqdm import tqdm
import os

# Environment checks
print(f" Torch version: {torch.__version__}")

import transformers
print(f" Transformers version: {transformers.__version__}")

# Check if CUDA is available and set device accordingly
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" Using device: {device}")

In [ ]:
# =========================
# Load Tokenizer & Model
# =========================

model_checkpoint = "microsoft/biogpt"  # or BioGPT-Large

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# BioGPT may not define a pad token → fix it
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load causal LM model
model = AutoModelForCausalLM.from_pretrained(model_checkpoint).to(device)

# Make sure model knows the pad token
model.config.pad_token_id = tokenizer.pad_token_id

print(f" Model and tokenizer loaded to device: {device}")

In [ ]:
from datasets import load_from_disk
train_path = "/content/drive/MyDrive/biomedical_text_generation/data/tokenized/biogpt_title_text_gen/train"
# Load FULL training dataset
full_train_dataset = load_from_disk(train_path)

# =========================
#  Create 90/10 Split
# =========================
split_dataset = full_train_dataset.train_test_split(
    test_size=0.1,   # 10% for validation
    seed=42,
    shuffle=True
)

train_dataset = split_dataset["train"]
validation_dataset = split_dataset["test"]

print(f" Train size: {len(train_dataset)}")
print(f" Validation size: {len(validation_dataset)}")

In [ ]:
training_args = TrainingArguments(
    output_dir=out_dir,

    evaluation_strategy="epoch",
    save_strategy="epoch",

    learning_rate=1e-5,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=32,  # effective batch size = 32

    num_train_epochs=100,
    weight_decay=0.01,
    save_total_limit=3,

    logging_dir="/content/drive/MyDrive/biomedical_text_generation/outputs/biogpt",
    logging_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    report_to="none",
    fp16=True,

    gradient_checkpointing=True
)

In [ ]:
# rouge_metric = evaluate.load("rouge")
# bleu_metric = evaluate.load("bleu")

# def compute_metrics(eval_pred):
#     predictions, labels = eval_pred

#     if isinstance(predictions, tuple):
#         predictions = predictions[0]

#     predictions = np.array(predictions)
#     labels = np.array(labels)

#     # If logits slip through, convert to ids
#     if predictions.ndim == 3:
#         predictions = predictions.argmax(-1)

#     # Replace ignore index in labels
#     labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

#     # Remove any negative ids (e.g., -1) before decoding
#     predictions = np.where(predictions < 0, tokenizer.pad_token_id, predictions)
#     labels      = np.where(labels < 0, tokenizer.pad_token_id, labels)

#     # Clamp to vocab range
#     vocab_size = len(tokenizer)
#     predictions = np.clip(predictions, 0, vocab_size - 1)
#     labels      = np.clip(labels, 0, vocab_size - 1)

#     decoded_preds = tokenizer.batch_decode(predictions.tolist(), skip_special_tokens=True)
#     decoded_labels = tokenizer.batch_decode(labels.tolist(), skip_special_tokens=True)

#     #  Fix None values
#     decoded_preds = [p if isinstance(p, str) else "" for p in decoded_preds]
#     decoded_labels = [l if isinstance(l, str) else "" for l in decoded_labels]

#     decoded_preds = [p.strip() for p in decoded_preds]
#     decoded_labels = [l.strip() for l in decoded_labels]

#     rouge = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)

#     bleu = bleu_metric.compute(
#         predictions=decoded_preds,
#         references=[[l] for l in decoded_labels]
#     )

#     rouge["bleu"] = bleu["bleu"]
#     return {k: round(v, 4) for k, v in rouge.items()}

In [ ]:
# # ============================
# # Trainer Initialization
# # ============================

# data_collator = DataCollatorForLanguageModeling(
#     tokenizer=tokenizer,
#     mlm=False
# )

# trainer = Trainer(
#     model=model,
#     args=training_args,

#     train_dataset=train_dataset,
#     eval_dataset=validation_dataset,

#     tokenizer=tokenizer,
#     data_collator=data_collator,

#     compute_metrics=None,

#     callbacks=[
#         EarlyStoppingCallback(early_stopping_patience=5)
#     ]
# )

In [ ]:
model.config.use_cache = False

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    tokenizer=tokenizer,
    compute_metrics=None,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
)

In [ ]:
# # ===============================
# # Start Fine-Tuning
# # ===============================
# train_result = trainer.train(resume_from_checkpoint=False)  # Starts the training loop using the trainer configuration

# # ===============================
# # Save Final Model & Tokenizer
# # ===============================
# # Save both the trained model and tokenizer for later use
# trainer.save_model("/content/drive/MyDrive/biomedical_text_generation/models/biogpt_title_text_gen_final")
# tokenizer.save_pretrained("/content/drive/MyDrive/biomedical_text_generation/models/biogpt_title_text_gen_final")

# print(" Fine-tuning completed and the final model has been saved.")


In [ ]:
import gc
import torch

del trainer
torch.cuda.empty_cache()
gc.collect()

In [ ]:
# ===============================
# Start Fine-Tuning
# ===============================

train_result = trainer.train(resume_from_checkpoint=False)

# ===============================
# Save Final Model & Tokenizer
# ===============================

final_dir = "/content/drive/MyDrive/biomedical_text_generation/models/biogpt_title_text_gen_final"

trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)

print(" Fine-tuning completed and the final model has been saved.")

trainer.log_metrics("train", train_result.metrics)
trainer.save_metrics("train", train_result.metrics)
trainer.save_state()